# Product Recommendation Engine
### Ecommerce AI Series - Nub Labs

**Playbook:** [nublabs.com/playbooks/ecommerce/recommendation-engine](https://nublabs.com/playbooks/ecommerce/recommendation-engine)  
**Reference platform:** [TechHeaven](https://github.com/Nub-Labs/techheaven-reference-platform) - a fictional ecommerce business running on Bagisto 2.x

---

TechHeaven carries 291 products across 40 categories. Every customer sees the same homepage. This notebook builds a recommendation engine that personalises product discovery using the transaction data TechHeaven already generates.

**What you will build:**
1. Association rules - what products are bought together, with support / confidence / lift
2. Collaborative filtering - recommendations based on similar customers' behaviour (SVD)
3. Content-based filtering - recommendations based on product attribute similarity (TF-IDF)
4. Hybrid blending - weighted combination of all three signals
5. Evaluation - Precision@K, Recall@K, NDCG@K on held-out test orders

**What you need:**
- Python virtualenv: `pip install -r requirements.txt`
- No database or API keys required - all data is fetched from GitHub

## 0. Setup

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install -q pandas numpy scipy scikit-learn mlxtend matplotlib seaborn
    print('Colab: packages installed')
else:
    print('Local: using virtualenv')

In [ ]:
import json
import urllib.request
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.float_format', '{:.4f}'.format)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

BASE = 'https://raw.githubusercontent.com/Nub-Labs/techheaven-reference-platform/main/data'

def fetch_json(url: str) -> list:
    with urllib.request.urlopen(url) as r:
        return json.load(r)

print('Imports ready')

## 1. Load Data

In [ ]:
print('Fetching TechHeaven data from GitHub...')

orders_raw      = fetch_json(f'{BASE}/transactions/orders.json')
items_raw       = fetch_json(f'{BASE}/transactions/order_items.json')
products_raw    = fetch_json(f'{BASE}/catalog/products.json')
reviews_raw     = fetch_json(f'{BASE}/reviews/reviews.json')

orders   = pd.DataFrame(orders_raw)
items    = pd.DataFrame(items_raw)
products = pd.DataFrame(products_raw)
reviews  = pd.DataFrame(reviews_raw)

orders['order_date'] = pd.to_datetime(orders['order_date'])
reviews['review_date'] = pd.to_datetime(reviews['review_date'])

print(f'Orders:         {len(orders):>6,}  ({orders["customer_id"].nunique():,} unique customers)')
print(f'Order items:    {len(items):>6,}  ({items["product_id"].nunique():,} unique products)')
print(f'Products:       {len(products):>6,}')
print(f'Reviews:        {len(reviews):>6,}')

## 2. Exploratory Data Analysis

In [ ]:
# Merge items with customer_id from orders
items_full = items.merge(orders[['order_id', 'customer_id', 'order_date']], on='order_id', how='inner')

print('=== Dataset summary ===')
print(f'Customers with orders:  {items_full["customer_id"].nunique():,}')
print(f'Products purchased:     {items_full["product_id"].nunique():,}')
print(f'Order-product pairs:    {len(items_full):,}')
print()

items_per_order = items_full.groupby('order_id')['product_id'].count()
orders_per_customer = items_full.groupby('customer_id')['order_id'].nunique()

print(f'Avg items per order:         {items_per_order.mean():.1f}  (median {items_per_order.median():.0f})')
print(f'Avg orders per customer:     {orders_per_customer.mean():.1f}  (median {orders_per_customer.median():.0f})')
print()

# Sparsity of user-item matrix
n_users    = items_full['customer_id'].nunique()
n_products = items_full['product_id'].nunique()
n_interactions = items_full.groupby(['customer_id', 'product_id']).ngroups
sparsity = 1 - n_interactions / (n_users * n_products)
print(f'User-item matrix:  {n_users} x {n_products}')
print(f'Non-zero cells:    {n_interactions:,}')
print(f'Sparsity:          {sparsity:.1%}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Items per order distribution
ax = axes[0]
items_per_order.clip(upper=10).value_counts().sort_index().plot(kind='bar', ax=ax, color='#7c3aed', edgecolor='white')
ax.set_title('Items per order')
ax.set_xlabel('Number of items')
ax.set_ylabel('Orders')
ax.tick_params(axis='x', rotation=0)

# Orders per customer distribution
ax = axes[1]
orders_per_customer.clip(upper=15).value_counts().sort_index().plot(kind='bar', ax=ax, color='#4f46e5', edgecolor='white')
ax.set_title('Orders per customer')
ax.set_xlabel('Number of orders')
ax.set_ylabel('Customers')
ax.tick_params(axis='x', rotation=0)

# Top categories by revenue
ax = axes[2]
cat_revenue = items_full.groupby('category')['price'].sum().nlargest(10)
cat_revenue.plot(kind='barh', ax=ax, color='#0ea5e9', edgecolor='white')
ax.set_title('Revenue by category (top 10)')
ax.set_xlabel('Total revenue ($)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}k'))

plt.tight_layout()
plt.show()

In [ ]:
# Product popularity: long-tail distribution
product_purchase_counts = items_full.groupby('product_id').size().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(len(product_purchase_counts)), product_purchase_counts.values, color='#7c3aed', alpha=0.7, width=1.0)
ax.set_xlabel('Product rank (by purchase count)')
ax.set_ylabel('Times purchased')
ax.set_title('Product purchase distribution - long tail')

top20_share = product_purchase_counts.head(20).sum() / product_purchase_counts.sum()
ax.axvline(20, color='red', linestyle='--', alpha=0.7, label=f'Top 20 products = {top20_share:.0%} of purchases')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Top 20 products account for {top20_share:.1%} of all purchases')
print(f'Remaining {len(product_purchase_counts)-20} products account for {1-top20_share:.1%}')

In [ ]:
# Rating distribution
fig, ax = plt.subplots(figsize=(7, 3))
reviews['rating'].value_counts().sort_index().plot(kind='bar', ax=ax, color='#f59e0b', edgecolor='white')
ax.set_title('Review rating distribution')
ax.set_xlabel('Rating (1-5 stars)')
ax.set_ylabel('Reviews')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

print(f'Average rating: {reviews["rating"].mean():.2f} / 5.0')
print(f'Reviewers:      {reviews["customer_id"].nunique():,}')
print(f'Products rated: {reviews["product_id"].nunique():,}')

## 3. Association Rules

Association rule mining finds products that are bought together more often than chance. The Apriori algorithm enumerates all frequent item sets - combinations that appear in at least `min_support` fraction of orders - then derives rules from them.

**Key metrics:**
- **Support(A, B)**: fraction of all orders that contain both A and B
- **Confidence(A -> B)**: P(B | A) - of orders containing A, what fraction also contain B
- **Lift(A -> B)**: confidence / P(B) - how much more likely B is given A vs. at random. Lift > 1 means a real relationship, not coincidence

In [ ]:
# Build transaction list: one list of product names per order
# Use product names (not IDs) so rules are human-readable
transactions = (
    items_full
    .groupby('order_id')['product_name']
    .apply(list)
    .tolist()
)

# Encode as boolean one-hot matrix
te = TransactionEncoder()
te_array = te.fit_transform(transactions)
df_te = pd.DataFrame(te_array, columns=te.columns_)

print(f'Transaction matrix: {df_te.shape[0]:,} orders x {df_te.shape[1]} unique products')
print(f'Average basket size: {df_te.sum(axis=1).mean():.1f} products')

In [ ]:
# Mine frequent itemsets
# min_support=0.005 means the pair must appear in at least 0.5% of orders
frequent_itemsets = apriori(df_te, min_support=0.005, use_colnames=True, max_len=2)

# Derive association rules with minimum lift of 1.5
rules = association_rules(
    frequent_itemsets,
    metric='lift',
    min_threshold=1.5,
    num_itemsets=len(frequent_itemsets),
)
rules = rules.sort_values('lift', ascending=False)

print(f'Frequent itemsets (support >= 0.5%):  {len(frequent_itemsets)}')
print(f'Rules (lift >= 1.5):                  {len(rules)}')

In [ ]:
# Top 15 rules by lift
top_rules = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(15).copy()
top_rules['antecedents'] = top_rules['antecedents'].apply(lambda x: ', '.join(list(x)))
top_rules['consequents'] = top_rules['consequents'].apply(lambda x: ', '.join(list(x)))
top_rules.columns = ['If customer buys', 'Also recommend', 'Support', 'Confidence', 'Lift']
top_rules

In [ ]:
# Visualise support vs confidence, sized by lift
fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(
    rules['support'],
    rules['confidence'],
    c=rules['lift'],
    s=rules['lift'] * 15,
    cmap='plasma',
    alpha=0.7,
    edgecolors='white',
    linewidth=0.5,
)
plt.colorbar(scatter, ax=ax, label='Lift')
ax.set_xlabel('Support (fraction of orders containing both products)')
ax.set_ylabel('Confidence P(B | A)')
ax.set_title('Association rules: support vs confidence vs lift')
plt.tight_layout()
plt.show()

In [ ]:
def recommend_associated(product_name: str, rules_df: pd.DataFrame, top_k: int = 5) -> pd.DataFrame:
    """Return top-k recommendations for a product using association rules."""
    # Match rules where the product appears as antecedent
    mask = rules_df['antecedents'].apply(lambda x: product_name in x)
    relevant = rules_df[mask].copy()
    relevant['recommendation'] = relevant['consequents'].apply(lambda x: list(x)[0])
    return relevant[['recommendation', 'confidence', 'lift']].head(top_k)


# Rebuild rules with frozenset antecedents for lookup
_rules = association_rules(
    frequent_itemsets, metric='lift', min_threshold=1.5,
    num_itemsets=len(frequent_itemsets)
).sort_values('lift', ascending=False)

test_products = [
    'Apple MacBook Air 15" M3',
    'Sony WH-1000XM5 Wireless Headphones',
]

for product in test_products:
    recs = recommend_associated(product, _rules)
    if len(recs):
        print(f'Customers who bought "{product}" also bought:')
        for _, row in recs.iterrows():
            print(f'  - {row["recommendation"]:55s}  confidence={row["confidence"]:.2f}  lift={row["lift"]:.2f}')
    else:
        print(f'No rules found for "{product}" at this support threshold')
    print()

## 4. Collaborative Filtering

Collaborative filtering answers: "what would a customer like, based on what customers similar to them have purchased?"

We use **matrix factorization via SVD** (Singular Value Decomposition). The user-item interaction matrix is decomposed into two lower-dimensional matrices (user latent factors and item latent factors) that capture hidden preference dimensions. Reconstructing the matrix fills in the missing entries - the predicted preference scores for products the user has not yet purchased.

The latent factors are not labelled, but they correspond to real behavioural patterns in the data: "audio enthusiast", "mobile professional", "gaming". SVD discovers these without being told about them.

In [ ]:
# Build user-item interaction matrix using purchase count as signal
# Rating-based: use reviews if available (explicit feedback)
# Purchase-based: use order items (implicit feedback)

# Merge reviews with customer info - use rating as explicit signal
ratings = reviews.merge(
    orders[['order_id', 'customer_id']].drop_duplicates('customer_id'),
    on='customer_id', how='inner'
)[['customer_id', 'product_id', 'rating']].drop_duplicates(['customer_id', 'product_id'])

# Fall back to implicit purchase signal for customers without reviews
purchase_signal = (
    items_full
    .groupby(['customer_id', 'product_id'])['qty']
    .sum()
    .clip(upper=5)  # cap at 5 to reduce outlier weight
    .reset_index()
    .rename(columns={'qty': 'signal'})
)

# Normalise ratings to 1-5 scale, purchase signal to 1-5 scale
combined = ratings.rename(columns={'rating': 'signal'})
# Add purchase signals for customer-product pairs not covered by reviews
rated_pairs = set(zip(combined['customer_id'], combined['product_id']))
extra = purchase_signal[
    ~purchase_signal.apply(lambda r: (r['customer_id'], r['product_id']) in rated_pairs, axis=1)
].copy()
# Scale purchase count (1-5) to match rating scale (1-5)
combined = pd.concat([combined, extra], ignore_index=True)

print(f'Total interaction signals: {len(combined):,}')
print(f'Unique customers:          {combined["customer_id"].nunique():,}')
print(f'Unique products:           {combined["product_id"].nunique():,}')

In [ ]:
# Train / test split: hold out the last order per customer
# Sort by order_date and take the final purchased product per customer as test
last_purchase = (
    items_full.sort_values('order_date')
    .groupby('customer_id')
    .last()
    .reset_index()[['customer_id', 'product_id']]
)
test_pairs = set(zip(last_purchase['customer_id'], last_purchase['product_id']))

train_signal = combined[
    ~combined.apply(lambda r: (r['customer_id'], r['product_id']) in test_pairs, axis=1)
]

print(f'Train signals: {len(train_signal):,}')
print(f'Test pairs:    {len(last_purchase):,}')

In [ ]:
# Build sparse user-item matrix from training data
user_ids   = sorted(train_signal['customer_id'].unique())
item_ids   = sorted(train_signal['product_id'].unique())
user_idx   = {u: i for i, u in enumerate(user_ids)}
item_idx   = {p: i for i, p in enumerate(item_ids)}
idx_item   = {i: p for p, i in item_idx.items()}  # reverse map for recommendations

rows = train_signal['customer_id'].map(user_idx)
cols = train_signal['product_id'].map(item_idx)
vals = train_signal['signal'].astype(float)

matrix = csr_matrix(
    (vals, (rows, cols)),
    shape=(len(user_ids), len(item_ids)),
    dtype=np.float32,
)

print(f'User-item matrix: {matrix.shape[0]} x {matrix.shape[1]}')
print(f'Non-zero:         {matrix.nnz:,}')
print(f'Sparsity:         {1 - matrix.nnz / (matrix.shape[0] * matrix.shape[1]):.1%}')

In [ ]:
# SVD decomposition: k=50 latent factors
# The matrix must be dense for scipy.sparse.linalg.svds
K = 50
dense = matrix.toarray()

# Mean-centre each user's ratings before decomposition
user_means = np.true_divide(dense.sum(axis=1), (dense != 0).sum(axis=1) + 1e-10)
dense_centered = dense.copy().astype(np.float32)
for i, mean in enumerate(user_means):
    dense_centered[i, dense[i] > 0] -= mean

U, sigma, Vt = svds(dense_centered, k=K)
sigma_diag = np.diag(sigma)

# Predicted ratings matrix
predicted = np.dot(np.dot(U, sigma_diag), Vt)
# Add back user means
predicted += user_means.reshape(-1, 1)

print(f'SVD complete: U{U.shape}, sigma{sigma.shape}, Vt{Vt.shape}')
print(f'Predicted matrix: {predicted.shape}')

In [ ]:
def recommend_cf(customer_id: int, n: int = 10, exclude_purchased: bool = True) -> list[tuple[int, float]]:
    """Return top-n (product_id, predicted_score) for a customer via SVD."""
    if customer_id not in user_idx:
        return []  # cold start - no history
    u = user_idx[customer_id]
    scores = predicted[u].copy()
    if exclude_purchased:
        purchased_cols = matrix[u].nonzero()[1]
        scores[purchased_cols] = -np.inf
    top_cols = np.argsort(scores)[::-1][:n]
    return [(idx_item[c], float(scores[c])) for c in top_cols if scores[c] > -np.inf]


# Demonstrate on a customer with multiple orders
demo_customer = items_full.groupby('customer_id')['order_id'].nunique().idxmax()
demo_history  = items_full[items_full['customer_id'] == demo_customer][['product_name', 'category']].drop_duplicates()

print(f'Customer {demo_customer} purchase history ({len(demo_history)} products):')
for _, row in demo_history.head(8).iterrows():
    print(f'  - {row["product_name"]:55s}  [{row["category"]}]')
print()

cf_recs = recommend_cf(demo_customer, n=8)
pid_to_name = dict(zip(products['product_id'], products['name']))
pid_to_cat  = dict(zip(products['product_id'], products['category']))

print(f'Collaborative filtering recommendations:')
for pid, score in cf_recs:
    name = pid_to_name.get(pid, f'product_{pid}')
    cat  = pid_to_cat.get(pid, '')
    print(f'  - {name:55s}  [{cat}]  score={score:.3f}')

In [ ]:
# Visualise the latent factor space: project to 2D via top 2 singular values
# Items with similar latent factor profiles cluster together
U2, s2, Vt2 = svds(dense_centered, k=2)
item_factors_2d = Vt2.T  # shape: (n_items, 2)

item_df = pd.DataFrame({
    'x': item_factors_2d[:, 0],
    'y': item_factors_2d[:, 1],
    'product_id': [idx_item[i] for i in range(len(item_ids))],
})
item_df['category'] = item_df['product_id'].map(pid_to_cat)
item_df['name'] = item_df['product_id'].map(pid_to_name)

fig, ax = plt.subplots(figsize=(12, 8))
categories = item_df['category'].dropna().unique()
palette = plt.cm.tab20(np.linspace(0, 1, min(len(categories), 20)))
cat_color = {cat: palette[i % 20] for i, cat in enumerate(categories)}

for cat, group in item_df.groupby('category'):
    ax.scatter(group['x'], group['y'], label=cat, alpha=0.7, s=40, color=cat_color.get(cat, 'grey'))

ax.set_title('Product latent factor space (SVD top 2 components)')
ax.set_xlabel('Latent factor 1')
ax.set_ylabel('Latent factor 2')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=7, markerscale=1.5)
plt.tight_layout()
plt.show()

## 5. Content-Based Filtering

Content-based filtering recommends products similar to ones the customer has already purchased, based on product attributes - not on other customers' behaviour.

We build a TF-IDF feature vector for each product from its name, brand, category, and description. Cosine similarity between these vectors measures product similarity. A customer who bought a Sony headphone gets recommended headphones with similar feature profiles.

**Key advantage:** new products are immediately recommendable. A new headphone added to the catalogue is similar to existing headphones from day one, without needing any purchase history.

In [ ]:
# Build product feature text: combine name, brand, category, description
# Weight important fields by repetition (category appears 3x to increase influence)
def build_features(row) -> str:
    parts = []
    if row.get('name'):     parts.append(str(row['name']))
    if row.get('brand'):    parts.extend([str(row['brand'])] * 2)
    if row.get('category'): parts.extend([str(row['category'])] * 3)
    if row.get('short_description'): parts.append(str(row['short_description']))
    if row.get('description'):
        # Truncate long descriptions to avoid drowning brand/category signal
        parts.append(str(row['description'])[:500])
    return ' '.join(parts)

products['features'] = products.apply(build_features, axis=1)

# TF-IDF: convert feature text to vectors
tfidf = TfidfVectorizer(
    max_features=8000,
    stop_words='english',
    ngram_range=(1, 2),
    min_df=1,
)
tfidf_matrix = tfidf.fit_transform(products['features'])

print(f'Product feature matrix: {tfidf_matrix.shape[0]} products x {tfidf_matrix.shape[1]} TF-IDF features')

In [ ]:
# Compute full cosine similarity matrix
content_sim = cosine_similarity(tfidf_matrix)

prod_id_to_idx = {int(pid): i for i, pid in enumerate(products['product_id'])}
idx_to_prod_id = {i: int(pid) for i, pid in enumerate(products['product_id'])}


def recommend_cbf(product_id: int, n: int = 8, exclude_same: bool = True) -> list[tuple[int, float]]:
    """Return top-n similar products by content (TF-IDF cosine similarity)."""
    if product_id not in prod_id_to_idx:
        return []
    idx = prod_id_to_idx[product_id]
    sim_scores = content_sim[idx].copy()
    if exclude_same:
        sim_scores[idx] = -1
    top_indices = np.argsort(sim_scores)[::-1][:n]
    return [(idx_to_prod_id[i], float(sim_scores[i])) for i in top_indices]


def recommend_cbf_for_customer(customer_id: int, n: int = 10, use_train_only: bool = False) -> list[tuple[int, float]]:
    """Aggregate CBF recommendations from a customer's purchase history.
    
    use_train_only=True during evaluation so the held-out test item is not
    included in 'purchased' (which would prevent it from being recommended).
    """
    source = train_signal if use_train_only else items_full
    purchased = set(source[source['customer_id'] == customer_id]['product_id'].unique())
    if not purchased:
        return []
    score_agg = defaultdict(float)
    for pid in purchased:
        for rec_pid, sim in recommend_cbf(pid, n=30):
            if rec_pid not in purchased:
                score_agg[rec_pid] += sim
    ranked = sorted(score_agg.items(), key=lambda x: x[1], reverse=True)
    return ranked[:n]


# Show similar products for a seed product
seed_name = 'Apple MacBook Air 15" M3'
seed_row  = products[products['name'] == seed_name].iloc[0]
seed_pid  = int(seed_row['product_id'])

print(f'Products similar to "{seed_name}":')
for pid, sim in recommend_cbf(seed_pid, n=8):
    row = products[products['product_id'] == pid].iloc[0]
    print(f'  {sim:.3f}  {row["name"]:55s}  [{row.get("category", "")}]')

In [ ]:
# Show content-based recommendations for the demo customer
cbf_recs = recommend_cbf_for_customer(demo_customer, n=8)
print(f'Content-based recommendations for customer {demo_customer}:')
for pid, score in cbf_recs:
    name = pid_to_name.get(pid, f'product_{pid}')
    cat  = pid_to_cat.get(pid, '')
    print(f'  {score:.3f}  {name:55s}  [{cat}]')

In [ ]:
# Visualise product similarity heatmap for a subset of products
# Pick top 20 products by purchase count for a readable heatmap
top_pids = product_purchase_counts.head(20).index.tolist()
top_pid_set  = set(top_pids)
top_products = products[products['product_id'].isin(top_pid_set)]
top_indices  = [prod_id_to_idx[int(pid)] for pid in top_products['product_id'] if int(pid) in prod_id_to_idx]
top_names    = [products.iloc[i]['name'][:30] for i in top_indices]

top_sim = content_sim[np.ix_(top_indices, top_indices)]

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    top_sim,
    xticklabels=top_names, yticklabels=top_names,
    cmap='YlOrRd', vmin=0, vmax=1,
    ax=ax, square=True, linewidths=0.3,
)
ax.set_title('Content-based similarity matrix (top 20 products)')
ax.tick_params(axis='x', rotation=45)
ax.tick_params(axis='y', rotation=0)
plt.tight_layout()
plt.show()

## 6. Hybrid Recommendation

In production, no single model is used in isolation. A weighted hybrid combines scores from multiple models to offset each approach's weaknesses.

The hybrid uses a **switching** component to handle cold start: when a customer has no purchase history (new user), the system falls back to category-level popularity. As history accumulates, collaborative filtering weight increases.

Weights are starting points - they should be tuned via A/B testing against business metrics.

In [ ]:
# Association rules as a product-level lookup dict
# key: antecedent product name -> list of (recommended_product_name, lift)
ar_lookup: dict[str, list[tuple[str, float]]] = defaultdict(list)
for _, row in _rules.iterrows():
    for ant in row['antecedents']:
        for con in row['consequents']:
            ar_lookup[ant].append((con, float(row['lift'])))

name_to_pid = dict(zip(products['name'], products['product_id'].astype(int)))


def recommend_hybrid(
    customer_id: int,
    n: int = 10,
    w_cf: float = 0.5,
    w_cbf: float = 0.3,
    w_ar: float = 0.2,
    use_train_only: bool = False,
) -> list[dict]:
    """
    Weighted hybrid recommendation.
    - Switching behaviour: reduces CF weight if customer has < 3 orders.
    - Falls back to category popularity for cold-start users.
    - use_train_only=True during evaluation to avoid leaking the held-out test item
      into the 'already purchased' filter (which would prevent it from appearing
      in recommendations and cause Precision@K = 0).
    """
    source = train_signal if use_train_only else items_full
    purchased_pids = set(source[source['customer_id'] == customer_id]['product_id'].unique())
    n_orders = orders[orders['customer_id'] == customer_id]['order_id'].nunique()

    # Adjust weights based on history depth
    if n_orders < 3:
        w_cf, w_cbf, w_ar = 0.1, 0.5, 0.4

    scores: dict[int, dict] = {}

    # Collaborative filtering scores
    for pid, score in recommend_cf(customer_id, n=n * 3):
        if pid not in purchased_pids:
            scores.setdefault(pid, {'cf': 0, 'cbf': 0, 'ar': 0})
            scores[pid]['cf'] = score

    # Content-based filtering scores
    for pid, score in recommend_cbf_for_customer(customer_id, n=n * 3, use_train_only=use_train_only):
        if pid not in purchased_pids:
            scores.setdefault(pid, {'cf': 0, 'cbf': 0, 'ar': 0})
            scores[pid]['cbf'] = score

    # Association rules: surface rules from purchased products
    for ppid in list(purchased_pids)[:20]:
        pname = pid_to_name.get(ppid, '')
        for rec_name, lift in ar_lookup.get(pname, [])[:5]:
            rec_pid = name_to_pid.get(rec_name)
            if rec_pid and rec_pid not in purchased_pids:
                scores.setdefault(rec_pid, {'cf': 0, 'cbf': 0, 'ar': 0})
                scores[rec_pid]['ar'] = max(scores[rec_pid]['ar'], min(lift / 5.0, 1.0))

    # Normalise each model's scores to [0, 1] before blending
    def norm(vals: list[float]) -> list[float]:
        mn, mx = min(vals), max(vals)
        if mx == mn:
            return [0.5] * len(vals)
        return [(v - mn) / (mx - mn) for v in vals]

    if scores:
        pids = list(scores.keys())
        cf_vals  = norm([scores[p]['cf']  for p in pids])
        cbf_vals = norm([scores[p]['cbf'] for p in pids])
        ar_vals  = norm([scores[p]['ar']  for p in pids])

        blended = {
            pid: w_cf * cf + w_cbf * cbf + w_ar * ar
            for pid, cf, cbf, ar in zip(pids, cf_vals, cbf_vals, ar_vals)
        }
        ranked = sorted(blended.items(), key=lambda x: x[1], reverse=True)[:n]
    else:
        # Cold start: popularity fallback
        popular = product_purchase_counts.head(n).index.tolist()
        ranked = [(pid, 0.0) for pid in popular if pid not in purchased_pids]

    results = []
    for pid, score in ranked:
        detail = scores.get(pid, {})
        reason = 'popularity'
        if detail.get('cf', 0) > 0 and detail.get('ar', 0) > 0:
            reason = 'bought together + similar customers'
        elif detail.get('cf', 0) > 0:
            reason = 'similar customers also bought'
        elif detail.get('cbf', 0) > 0:
            reason = 'similar to your purchases'
        elif detail.get('ar', 0) > 0:
            reason = 'frequently bought together'
        results.append({
            'product_id': pid,
            'name':       pid_to_name.get(pid, f'product_{pid}'),
            'category':   pid_to_cat.get(pid, ''),
            'score':      round(score, 4),
            'reason':     reason,
        })
    return results


# Run hybrid for the demo customer (inference mode - uses full history)
hybrid_recs = recommend_hybrid(demo_customer, n=10)
print(f'Hybrid recommendations for customer {demo_customer}:')
print(f'{"Product":<55} {"Category":<20} {"Score":>6}  Reason')
print('-' * 110)
for r in hybrid_recs:
    print(f'{r["name"]:<55} {r["category"]:<20} {r["score"]:>6.4f}  {r["reason"]}')

## 7. Evaluation

Offline evaluation measures how well each model predicts held-out purchases. We use:

- **Precision@K**: of the top-K recommendations, what fraction did the customer actually buy?
- **Recall@K**: of all the things the customer actually bought, what fraction appeared in the top-K?
- **NDCG@K** (Normalised Discounted Cumulative Gain): accounts for rank position - a relevant product at position 1 scores higher than the same product at position 5

**Interpretation:** Precision and Recall are in tension. A model can achieve perfect Recall@K by recommending everything. A model can achieve perfect Precision@1 by only recommending the single most likely product. NDCG balances both and rewards models that rank relevant items near the top.

In [ ]:
def precision_at_k(recommended: list, relevant: set, k: int) -> float:
    top_k = recommended[:k]
    return len(set(top_k) & relevant) / k


def recall_at_k(recommended: list, relevant: set, k: int) -> float:
    if not relevant:
        return 0.0
    top_k = recommended[:k]
    return len(set(top_k) & relevant) / len(relevant)


def ndcg_at_k(recommended: list, relevant: set, k: int) -> float:
    top_k = recommended[:k]
    dcg  = sum(1 / np.log2(i + 2) for i, item in enumerate(top_k) if item in relevant)
    idcg = sum(1 / np.log2(i + 2) for i in range(min(k, len(relevant))))
    return dcg / idcg if idcg > 0 else 0.0


K_VALUES = [5, 10, 20]

results = {model: {k: {'precision': [], 'recall': [], 'ndcg': []} for k in K_VALUES}
           for model in ['CF', 'CBF', 'Hybrid']}

for _, row in last_purchase.iterrows():
    cid      = row['customer_id']
    true_pid = int(row['product_id'])
    relevant = {true_pid}

    cf_pids  = [pid for pid, _ in recommend_cf(cid, n=max(K_VALUES))]
    # use_train_only=True: exclude held-out test item from the 'purchased' filter
    # so CBF and Hybrid can actually recommend it
    cbf_pids = [pid for pid, _ in recommend_cbf_for_customer(cid, n=max(K_VALUES), use_train_only=True)]
    hyb_pids = [r['product_id'] for r in recommend_hybrid(cid, n=max(K_VALUES), use_train_only=True)]

    for k in K_VALUES:
        for model, pids in [('CF', cf_pids), ('CBF', cbf_pids), ('Hybrid', hyb_pids)]:
            results[model][k]['precision'].append(precision_at_k(pids, relevant, k))
            results[model][k]['recall'].append(recall_at_k(pids, relevant, k))
            results[model][k]['ndcg'].append(ndcg_at_k(pids, relevant, k))

print('Evaluation complete.')

In [ ]:
# Summary table
rows_eval = []
for model in ['CF', 'CBF', 'Hybrid']:
    for k in K_VALUES:
        m = results[model][k]
        rows_eval.append({
            'Model': model,
            'K': k,
            'Precision@K': np.mean(m['precision']),
            'Recall@K':    np.mean(m['recall']),
            'NDCG@K':      np.mean(m['ndcg']),
        })

eval_df = pd.DataFrame(rows_eval)
eval_df

In [ ]:
# Plot evaluation metrics comparison at K=10
k10 = eval_df[eval_df['K'] == 10].set_index('Model')

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
colors = {'CF': '#4f46e5', 'CBF': '#7c3aed', 'Hybrid': '#0ea5e9'}

for ax, metric in zip(axes, ['Precision@K', 'Recall@K', 'NDCG@K']):
    vals = k10[metric]
    bars = ax.bar(vals.index, vals.values, color=[colors[m] for m in vals.index], edgecolor='white')
    ax.set_title(f'{metric} (K=10)')
    ax.set_ylim(0, max(vals.values) * 1.3)
    for bar, val in zip(bars, vals.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001,
                f'{val:.4f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Recommendation model comparison at K=10', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## 8. Business Insights

The mined association rules produce interpretable patterns that directly map to revenue opportunities. Each rule has a specific placement: high-lift rules go into checkout "frequently bought together" widgets; high-confidence rules go into post-purchase email sequences.

In [ ]:
# Top rules by category pair - which category combinations drive strongest associations
rules_named = _rules.copy()
rules_named['ant_name'] = rules_named['antecedents'].apply(lambda x: list(x)[0])
rules_named['con_name'] = rules_named['consequents'].apply(lambda x: list(x)[0])
rules_named['ant_cat']  = rules_named['ant_name'].map(lambda n: items_full[items_full['product_name'] == n]['category'].mode().iloc[0] if (items_full['product_name'] == n).any() else '')
rules_named['con_cat']  = rules_named['con_name'].map(lambda n: items_full[items_full['product_name'] == n]['category'].mode().iloc[0] if (items_full['product_name'] == n).any() else '')

cat_pair_lift = (
    rules_named[rules_named['ant_cat'] != rules_named['con_cat']]
    .groupby(['ant_cat', 'con_cat'])['lift']
    .mean()
    .reset_index()
    .rename(columns={'ant_cat': 'If customer buys category', 'con_cat': 'Recommend category', 'lift': 'Avg lift'})
    .sort_values('Avg lift', ascending=False)
    .head(15)
)
cat_pair_lift

In [ ]:
# Revenue opportunity: for each strong rule, estimate lift in revenue
# Orders containing rule antecedent but NOT consequent = addressable with recommendation
strong_rules = rules_named[rules_named['lift'] >= 2.0].copy()

revenue_opps = []
for _, rule in strong_rules.head(10).iterrows():
    ant = rule['ant_name']
    con = rule['con_name']
    con_price = items_full[items_full['product_name'] == con]['price'].median()

    # Orders with antecedent but without consequent
    orders_with_ant = set(items_full[items_full['product_name'] == ant]['order_id'])
    orders_with_con = set(items_full[items_full['product_name'] == con]['order_id'])
    missed_orders   = orders_with_ant - orders_with_con

    # Estimated revenue if confidence% of missed orders convert
    estimated_rev = len(missed_orders) * rule['confidence'] * (con_price or 0)
    revenue_opps.append({
        'Antecedent':         ant[:40],
        'Recommend':          con[:40],
        'Lift':               round(rule['lift'], 2),
        'Confidence':         round(rule['confidence'], 2),
        'Missed orders':      len(missed_orders),
        'Est. revenue ($)':   round(estimated_rev, 0),
    })

pd.DataFrame(revenue_opps).sort_values('Est. revenue ($)', ascending=False)

In [ ]:
# Review sentiment by category - average rating per category
reviews_with_cat = reviews.merge(
    items_full[['product_id', 'category']].drop_duplicates(),
    on='product_id', how='left'
)

cat_ratings = (
    reviews_with_cat.groupby('category')['rating']
    .agg(['mean', 'count'])
    .query('count >= 20')
    .sort_values('mean', ascending=False)
    .head(15)
)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(cat_ratings.index, cat_ratings['mean'], color='#f59e0b', edgecolor='white')
ax.set_xlim(3, 5)
ax.set_xlabel('Average rating')
ax.set_title('Average product rating by category (min 20 reviews)')
ax.axvline(reviews['rating'].mean(), color='grey', linestyle='--', alpha=0.7, label=f'Overall avg: {reviews["rating"].mean():.2f}')
ax.legend()
plt.tight_layout()
plt.show()

## 9. Production Considerations

**Scalability**  
Full matrix SVD is O(min(m,n)^3). At 1M users x 100K products, dense SVD is infeasible. Solutions: Alternating Least Squares (ALS) with sparse matrices, Approximate Nearest Neighbour (ANN) libraries (FAISS, ScaNN) for retrieval, and two-stage pipelines (candidate generation -> re-ranking).

**Feature stores**  
In production, user embeddings and item embeddings are pre-computed and stored in a feature store (Redis, Feast, Tecton). The serving layer performs a vector similarity lookup, not a matrix multiply, at inference time.

**Implicit vs explicit feedback**  
Explicit feedback (ratings) is scarce. Implicit feedback (views, add-to-cart, purchases, dwell time) is abundant but noisier. Libraries like `implicit` (ALS for implicit feedback) are purpose-built for ecommerce-scale implicit signal.

**Online learning**  
Batch-trained models are stale. Real-time event streams (Kafka, Kinesis) update user representations incrementally. Bandit algorithms (Thompson Sampling, UCB) balance exploration (showing new products) with exploitation (showing proven recommendations).

**A/B testing**  
Offline metrics (Precision@K, NDCG@K) are necessary but not sufficient. A model that scores higher offline may not convert better online. Always gate production changes behind A/B tests measuring add-to-cart rate, revenue per session, and return rate.

In [ ]:
# Final summary
print('=== TechHeaven Recommendation Engine - Summary ===')
print()
print('Dataset')
print(f'  Customers:        {items_full["customer_id"].nunique():,}')
print(f'  Products:         {items_full["product_id"].nunique():,}')
print(f'  Interactions:     {len(items_full):,}')
print(f'  Reviews:          {len(reviews):,}')
print()
print('Models trained')
print(f'  Association rules:  {len(_rules)} rules (lift >= 1.5, support >= 0.5%)')
print(f'  Collaborative:      SVD, k={K} latent factors')
print(f'  Content-based:      TF-IDF, {tfidf_matrix.shape[1]} features')
print(f'  Hybrid:             weighted blend, switching on history depth')
print()
print('Evaluation at K=10')
for model in ['CF', 'CBF', 'Hybrid']:
    m = results[model][10]
    print(f'  {model:<8} Precision={np.mean(m["precision"]):.4f}  Recall={np.mean(m["recall"]):.4f}  NDCG={np.mean(m["ndcg"]):.4f}')

## 10. Exercises

1. **Tune association rule thresholds**: Lower `min_support` to 0.001. How many more rules are generated? What happens to their interpretability?

2. **Vary SVD latent factors**: Run SVD with k=10, 25, 50, 100. Plot NDCG@10 against k. Where does accuracy plateau?

3. **Implicit vs explicit signal**: Replace the purchase+review combined signal with purchase-only implicit feedback (qty ordered). How does evaluation performance change?

4. **Cold start comparison**: Select 50 customers with only 1 order. Compare CF vs CBF vs popularity baseline on these customers. Which approach handles cold start better?

5. **Hybrid weight tuning**: Try weight combinations (CF=0.7, CBF=0.2, AR=0.1) and (CF=0.2, CBF=0.4, AR=0.4). Which combination produces the highest NDCG@10?

6. **Revenue impact estimate**: For the top 5 association rules by estimated revenue (section 8), calculate how many banner impressions would be needed at a 2% CTR and 10% add-to-cart rate to achieve that revenue.

## Further Reading

- Koren, Y., Bell, R., Volinsky, C. (2009). *Matrix Factorization Techniques for Recommender Systems*. IEEE Computer.
- He, R., McAuley, J. (2016). *Ups and Downs: Modeling the Visual Evolution of Fashion Trends with One-Class Collaborative Filtering*. WWW.
- Agrawal, R., Srikant, R. (1994). *Fast Algorithms for Mining Association Rules*. VLDB.
- `implicit` library: ALS for large-scale implicit feedback - github.com/benfred/implicit
- FAISS: billion-scale ANN search - github.com/facebookresearch/faiss
- Evidently AI: recommendation system monitoring - evidentlyai.com